# Ch 2. LLM이란 무엇인가

## 이 챕터에서 배우는 것

- LLM은 결국 **"다음 단어 맞히기"를 반복하는 기계**라는 한 줄 직관
- **토큰 · 컨텍스트 윈도우 · 온도 · 시스템 프롬프트** — 앞으로 모든 챕터에서 쓰일 4개 용어의 감각
- 파이썬 10줄로 첫 호출을 해보고, 응답이 **한 글자씩 어떻게 만들어지는지** 눈으로 확인한다
- 왜 LLM이 **때때로 거짓말을 하는가**(hallucination)에 대한 구조적 이해

원본 챕터: [LLM이란 무엇인가](https://desty.github.io/study-ai-assistant-engineering/part1/02-what-is-llm/)

In [ ]:
# 환경 설치
!pip install -q anthropic pandas

In [ ]:
import os
from getpass import getpass
os.environ.setdefault('ANTHROPIC_API_KEY', getpass('ANTHROPIC_API_KEY: '))

## 1. 시작 — 당신은 이미 LLM의 친척을 쓰고 있다

스마트폰에 **"안녕하세"** 까지 치면 "요"가 제안됩니다. 검색창에 **"오늘 날"** 까지 치면 "씨"가 튀어나옵니다. 이메일 앱은 **"수고하"** 뒤에 "셨습니다"를 붙여줍니다.

이 기능들은 모두 같은 원리로 동작합니다.

> **지금까지 본 텍스트 다음에 올 가능성이 가장 높은 글자/단어를 예측한다.**

## 2. 토큰 — 글자도 단어도 아닌 것

LLM은 "글자"나 "단어"가 아니라 **토큰(token)** 이라는 단위로 텍스트를 읽고 씁니다.

In [ ]:
# 토큰화 예시
korean_text = "안녕하세요"
english_text = "Hello"

print(f"한국어 '안녕하세요' 글자 수: {len(korean_text)}")
print(f"영어 'Hello' 글자 수: {len(english_text)}")
print()
print("주의: 실제 토큰 수는 모델의 tokenizer에 따라 다릅니다.")
print("한국어는 보통 영어보다 토큰이 1.5~2배 더 나옵니다.")

## 3. 다음 토큰은 어떻게 "골라지나"

LLM의 내부 작동을 아주 단순화하면 다음 토큰 생성 과정은 반복됩니다:

1. 입력 토큰들을 받음
2. 다음 올 토큰들의 확률 분포를 계산
3. 온도와 샘플링 전략에 따라 하나를 선택
4. 선택된 토큰을 입력 뒤에 이어붙임 → 다시 1번부터

In [ ]:
import pandas as pd

# 단어 생성 과정을 직관적으로 표현
steps = [
    {"스텝": 1, "입력": "오늘 점심은", "생성된 토큰": "김치찌개"},
    {"스텝": 2, "입력": "오늘 점심은 김치찌개", "생성된 토큰": "가"},
    {"스텝": 3, "입력": "오늘 점심은 김치찌개가", "생성된 토큰": "좋겠다"},
    {"스텝": 4, "입력": "오늘 점심은 김치찌개가 좋겠다", "생성된 토큰": "."},
]

df_steps = pd.DataFrame(steps)
print(df_steps.to_string(index=False))
print("\n우리가 보는 '한 문장 응답'은 사실 이 루프가 수십~수백 번 돈 결과입니다.")

## 4. 컨텍스트 윈도우

모델은 무한한 텍스트를 보지 못합니다. 한 번에 볼 수 있는 토큰 수가 정해져 있고, 이를 **컨텍스트 윈도우**라고 부릅니다.

In [ ]:
# 모델별 컨텍스트 윈도우
models = [
    {"모델 계열": "소형·빠른 모델", "컨텍스트": "8K~32K", "감각": "긴 대화 한두 번"},
    {"모델 계열": "중형 (대부분의 챗봇)", "컨텍스트": "128K", "감각": "책 한 권의 핵심"},
    {"모델 계열": "대형·고컨텍스트", "컨텍스트": "1M+", "감각": "책 여러 권 또는 대규모 코드"},
]

df_models = pd.DataFrame(models)
print(df_models.to_string(index=False))

## 5. 온도(Temperature) — 창의성 다이얼

같은 질문을 해도 LLM이 어떤 날은 평범하게, 어떤 날은 창의적으로 답하는 뒤에는 **온도(temperature)** 라는 설정값이 있습니다.

## 6. 시스템 프롬프트 — 모델에게 주는 "업무 지침"

LLM을 처음 만지면 `messages`에 `"role": "user"` 만 있는 예제를 보게 됩니다. 실전에서는 거의 항상 `"role": "system"` 메시지가 하나 더 있습니다.

## 8. 실습 — 파이썬 10줄로 첫 호출

In [ ]:
from anthropic import Anthropic

client = Anthropic()

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    system="당신은 친절한 한국어 설명 도우미입니다. 3줄 이내로 답하세요.",
    messages=[
        {"role": "user", "content": "LLM을 초등학생에게 설명해줘"}
    ],
)

print(response.content[0].text)

## 9. 손으로 감 잡기 — 파라미터 놀이

같은 질문을 **온도만 바꿔서** 여러 번 호출해봅니다.

In [ ]:
from anthropic import Anthropic
client = Anthropic()

for temp in [0.0, 0.7, 1.2]:
    print(f"\n=== temperature = {temp} ===")
    for i in range(3):
        r = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=60,
            temperature=temp,
            messages=[{"role": "user", "content": "점심 메뉴 하나 추천해줘. 한 줄로."}],
        )
        print(f"{i+1}. {r.content[0].text.strip()}")

**관찰 포인트**:

- 온도 0에서 3번 호출한 결과가 **거의 같은가**?
- 온도 1.2에서는 **얼마나 다양한가**?
- 가끔 이상한 답이 나오는 건 언제?

## Hallucination 유도 예시

LLM이 "모르는" 것도 그럴듯하게 지어내는 현상을 확인해봅시다.

In [ ]:
question = "2050년 서울의 인구는 얼마나 될까?"

print(f"질문: {question}")
print()

# 시스템 프롬프트 없이 호출 (위험)
print("=== 안전장치 없음 ===")
r1 = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=100,
    messages=[{"role": "user", "content": question}],
)
print(r1.content[0].text)
print()

# 안전장치 있음
print("=== 안전장치 있음 ===")
r2 = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=100,
    system="미래 예측은 할 수 없습니다. 모르는 것은 '확인이 필요합니다'라고 답하세요.",
    messages=[{"role": "user", "content": question}],
)
print(r2.content[0].text)

## 다음 장

여기까지 이해하면 이제 "내 Assistant는 어떤 블록들로 구성돼야 하는가"를 설계할 준비가 됐습니다.

[Ch 3. AI Assistant 시스템 개요](ch03_assistant_overview.ipynb) 로 이동